In [1]:
import pandas as pd
import geopandas as gpd
from unidecode import unidecode

In [2]:
file_path = "../data/raw/soil/soilmap.geojson" 
gdf = gpd.read_file(file_path)

In [3]:
gdf.head()

,gid,domsoil,faosoil,Type,geometry
0,1,Ao,Ao90-2/3c,"Orthic Acrisols within 90cm, medium and fine t...","MULTIPOLYGON (((106.66288 8.76057, 106.66295 8..."
1,2,Je,Je73-3a,"Eutric Fluvisols within 73 cm, fine textured, ...","MULTIPOLYGON (((107.15538 10.44863, 107.15474 ..."
2,3,Je,Je73-3a,"Eutric Fluvisols within 73 cm, fine textured, ...","MULTIPOLYGON (((107.14923 10.45511, 107.14891 ..."
3,4,Je,Je73-3a,"Eutric Fluvisols within 73 cm, fine textured, ...","MULTIPOLYGON (((106.9686 10.45353, 106.96735 1..."
4,5,Ao,Ao90-2/3c,"Orthic Acrisols within 90cm, medium and fine t...","MULTIPOLYGON (((109.21996 11.77342, 109.21888 ..."


In [4]:
print(gdf.crs)

EPSG:4326


In [5]:
print(gdf['faosoil'].value_counts())

faosoil
Ao107-2bc     42
Ao90-2/3c     27
Af60-1/2ab    23
Ge56-3a       18
Gd29-3a       15
Af61-1/2ab    15
Re83-1ab      14
Vp64-3a       10
Ag16-2a        8
Je73-3a        7
Ge55-3a        7
I-Lc-Bk-c      7
Fr33-3ab       7
I-Af-3c        6
Jt13-3a        6
Jt14-3a        5
Ag17-1/2ab     3
Je72-2a        2
Lc100-c        2
Lc99-2b        2
Af63-3c        1
Fa14-3ab       1
Fo102-3ab      1
Od21-a         1
Name: count, dtype: int64


In [6]:
# Function to simplify complex FAO codes into major soil groups
def simplify_fao_soil(code):
    if pd.isna(code):
        return "Unknown"
    code_str = str(code).upper()
    
    if code_str.startswith("AO") or code_str.startswith("AG"):
        return "Acrisols (Red-Yellow / Acid)"
    elif code_str.startswith("AF"):
        return "Ferralsols (Basaltic / Uplands)"
    elif code_str.startswith("GE") or code_str.startswith("GD") or code_str.startswith("FL"):
        return "Fluvisols / Gleysols (Alluvial / Lowlands)"
    elif code_str.startswith("RE") or code_str.startswith("LP") or code_str.startswith("LE"):
        return "Leptosols / Arenosols (Mountainous / Sandy)"
    elif code_str.startswith("HS") or code_str.startswith("VP"):
        return "Histosols / Organics (Peat)"
    else:
        return "Other / Complex"

# Apply mapping to create a clean categorical column
gdf['major_soil_type'] = gdf['faosoil'].apply(simplify_fao_soil)

print(gdf['major_soil_type'].value_counts())

major_soil_type
Acrisols (Red-Yellow / Acid)                   80
Other / Complex                                47
Fluvisols / Gleysols (Alluvial / Lowlands)     40
Ferralsols (Basaltic / Uplands)                39
Leptosols / Arenosols (Mountainous / Sandy)    14
Histosols / Organics (Peat)                    10
Name: count, dtype: int64


## Province Boundaries

In [7]:
# Load Vietnam province boundaries
provinces_gdf = gpd.read_file("../data/raw/soil/geoBoundaries-VNM-ADM1.geojson")

provinces_gdf.head()

,shapeName,shapeISO,shapeID,shapeGroup,shapeType,geometry
0,An Giang,VN-44,63759600B46141220358660,VNM,ADM1,"POLYGON ((105.18476 10.91406, 105.17936 10.924..."
1,Bà Rịa–Vũng Tàu,VN-43,63759600B21811289888982,VNM,ADM1,"POLYGON ((107.00723 10.60597, 107.01135 10.603..."
2,Bắc Giang,VN-54,63759600B37262939240335,VNM,ADM1,"POLYGON ((106.61213 21.26179, 106.618 21.26062..."
3,Bắc Kạn,VN-53,63759600B29118141744967,VNM,ADM1,"POLYGON ((105.53104 22.63266, 105.53185 22.625..."
4,Bạc Liêu,VN-55,63759600B49404698723526,VNM,ADM1,"POLYGON ((105.54956 9.61505, 105.52028 9.6145,..."


In [8]:
# spatial join (intersects)
joined_gdf = gpd.sjoin(gdf, provinces_gdf, how="inner", predicate="intersects")

# Find the most frequent (dominant) soil type for each province
dominant_soil = (
    joined_gdf.groupby(['shapeName', 'faosoil'])
    .size()
    .reset_index(name='count')
    .sort_values(['shapeName', 'count'], ascending=[True, False])
    .drop_duplicates(subset=['shapeName'])
)

print(dominant_soil.head(10))

          shapeName     faosoil  count
1          An Giang     Ge56-3a      2
7   Bà Rịa–Vũng Tàu     Je73-3a      3
11       Bình Dương     Ag16-2a      2
14       Bình Phước     Ag16-2a      3
18       Bình Thuận  Af60-1/2ab      3
30        Bình Định     Gd29-3a      8
31         Bạc Liêu     Ge55-3a      2
35        Bắc Giang  Af60-1/2ab      1
40          Bắc Kạn   Ao107-2bc      1
42         Bắc Ninh  Ag17-1/2ab      1


In [9]:
# 1. Comprehensive mapping dictionary for your exact FAO codes
fao_code_mapping = {
    'Ge56-3a': 'Gleysols (Lowland Alluvial / Wet)',
    'Ge55-3a': 'Gleysols (Lowland Alluvial / Wet)',
    'Gd29-3a': 'Dystric Gleysols (Alluvial)',
    'Je73-3a': 'Luvisols (High-Nutrient Clay)',
    'Ag16-2a': 'Acrisols (Red-Yellow / Degraded Clay)',
    'Ag17-1/2ab': 'Acrisols (Red-Yellow / Sloped)',
    'Ao107-2bc': 'Orthic Acrisols (Acid Red-Yellow)',
    'Af60-1/2ab': 'Ferralsols (Basaltic / Uplands)'
}

# 2. Map the codes to clean English text
joined_gdf['soil_type_en'] = joined_gdf['faosoil'].map(fao_code_mapping)

# Fallback in case any unmapped codes show up for other provinces
joined_gdf['soil_type_en'] = joined_gdf['soil_type_en'].fillna('Other Mineral Soils')

print(joined_gdf[['shapeName', 'faosoil', 'soil_type_en']])

           shapeName    faosoil                       soil_type_en
0            Côn Đảo  Ao90-2/3c                Other Mineral Soils
1    Bà Rịa–Vũng Tàu    Je73-3a      Luvisols (High-Nutrient Clay)
2    Bà Rịa–Vũng Tàu    Je73-3a      Luvisols (High-Nutrient Clay)
5          Khánh Hòa  Ao90-2/3c                Other Mineral Soils
7          Khánh Hòa    Ge56-3a  Gleysols (Lowland Alluvial / Wet)
..               ...        ...                                ...
229          Bắc Kạn  Ao107-2bc  Orthic Acrisols (Acid Red-Yellow)
229         Lạng Sơn  Ao107-2bc  Orthic Acrisols (Acid Red-Yellow)
229         Cao Bằng  Ao107-2bc  Orthic Acrisols (Acid Red-Yellow)
229          Lào Cai  Ao107-2bc  Orthic Acrisols (Acid Red-Yellow)
229         Hà Giang  Ao107-2bc  Orthic Acrisols (Acid Red-Yellow)

[425 rows x 3 columns]


In [10]:
clean_soil_df = joined_gdf[['shapeName', 'soil_type_en']].copy()
clean_soil_df.columns = ['province_name', 'soil']

# Drop duplicate rows (if a province appeared multiple times from the spatial join)
clean_soil_df = clean_soil_df.drop_duplicates(subset=['province_name'])

# Filter out any non-province entries if necessary (e.g., Côn Đảo is part of Bà Rịa-Vũng Tàu)
clean_soil_df = clean_soil_df[clean_soil_df['province_name'] != 'Côn Đảo']

# Convert provincial names to standard ASCII English (removes accents/diacritics)
clean_soil_df['province_name'] = clean_soil_df['province_name'].apply(lambda x: unidecode(str(x)))

# Clean up formatting (e.g., standardizing hyphens and spaces)
clean_soil_df['province_name'] = clean_soil_df['province_name'].str.replace('–', '-', regex=False)

# Drop duplicates and filter out standalone islands like Con Dao if required
clean_soil_df = clean_soil_df.drop_duplicates(subset=['province_name'])

# Reset index
clean_soil_df = clean_soil_df.reset_index(drop=True)

print(clean_soil_df.head(10))

     province_name                               soil
0  Ba Ria-Vung Tau      Luvisols (High-Nutrient Clay)
1        Khanh Hoa                Other Mineral Soils
2          Phu Yen                Other Mineral Soils
3        Binh Dinh        Dystric Gleysols (Alluvial)
4        Hai Phong  Orthic Acrisols (Acid Red-Yellow)
5       Quang Ninh  Orthic Acrisols (Acid Red-Yellow)
6         Lai Chau                Other Mineral Soils
7          Lao Cai                Other Mineral Soils
8          Ha Tinh                Other Mineral Soils
9          Nghe An                Other Mineral Soils


In [11]:
clean_soil_df['province_name'].nunique()

63

In [18]:
unique_provinces = clean_soil_df['province_name'].unique()
unique_provinces

<ArrowStringArray>
['Ba Ria-Vung Tau',       'Khanh Hoa',         'Phu Yen',       'Binh Dinh',
       'Hai Phong',      'Quang Ninh',        'Lai Chau',         'Lao Cai',
         'Ha Tinh',         'Nghe An',       'Thanh Hoa',      'Quang Binh',
        'Hoa Binh',          'Son La',         'Phu Tho',       'Dien Bien',
         'Yen Bai',        'Bac Ninh',       'Bac Giang',     'Thai Nguyen',
        'Ha Noi\t',       'Vinh Phuc',       'Thai Binh',          'Ha Nam',
        'Hung Yen',       'Hai Duong',       'Ninh Binh',        'Nam Dinh',
       'Quang Tri',  'Thua Thien Hue',       'Quang Nam',         'Da Nang',
      'Quang Ngai',         'Gia Lai',         'Kon Tum',         'Dak Lak',
        'Dak Nong',      'Binh Thuan',        'Dong Nai',      'Ninh Thuan',
        'Lam Dong',      'Binh Phuoc',     'Ho Chi Minh',      'Binh Duong',
        'Tay Ninh',         'Long An',       'Dong Thap',         'Ben Tre',
       'Vinh Long',      'Tien Giang',         'Can Tho',

## Add coordinates

In [13]:
province_coords = {
    # Red River Delta
    'Ha Noi': (21.0285, 105.8542), 'Ha Nam': (20.5833, 105.9167),
    'Hai Duong': (20.9367, 106.3314), 'Hai Phong': (20.8449, 106.6881),
    'Hung Yen': (20.6464, 106.0511), 'Nam Dinh': (20.4357, 106.1684),
    'Ninh Binh': (20.2581, 105.9797), 'Thai Binh': (20.4463, 106.3366),
    'Vinh Phuc': (21.3089, 105.6042), 'Bac Ninh': (21.1861, 106.0763),
    
    # Northern Midlands and Mountains
    'Ha Giang': (22.8233, 104.9839), 'Cao Bang': (22.6667, 106.2500),
    'Bac Kan': (22.1500, 105.8333), 'Tuyen Quang': (21.8233, 105.2156),
    'Lao Cai': (22.4856, 103.9704), 'Dien Bien': (21.3833, 103.0167),
    'Lai Chau': (22.3997, 103.4750), 'Son La': (21.3281, 103.9039),
    'Yen Bai': (21.7167, 104.8667), 'Hoa Binh': (20.8133, 105.3383),
    'Thai Nguyen': (21.5940, 105.8481), 'Lang Son': (21.8500, 106.7333),
    'Bac Giang': (21.2731, 106.1947), 'Phu Tho': (21.3250, 105.3986),
    
    # North Central and Central Coastal Areas
    'Thanh Hoa': (19.8067, 105.7852), 'Nghe An': (18.6796, 105.6813),
    'Ha Tinh': (18.3429, 105.9055), 'Quang Binh': (17.4777, 106.6025),
    'Quang Tri': (16.7499, 107.1856), 'Thua Thien Hue': (16.4637, 107.5909),
    'Da Nang': (16.0544, 108.2022), 'Quang Nam': (15.5558, 108.4725),
    'Quang Ngai': (15.1206, 108.7922), 'Binh Dinh': (13.7820, 109.2197),
    'Phu Yen': (13.0884, 109.3097), 'Khanh Hoa': (12.2388, 109.1967),
    'Ninh Thuận': (11.5643, 108.9886), 'Ninh Thuan': (11.5643, 108.9886),
    'Binh Thuận': (10.9275, 108.0987), 'Binh Thuan': (10.9275, 108.0987),
    
    # Central Highlands (Tay Nguyen)
    'Kon Tum': (14.3542, 108.0005), 'Gia Lai': (13.9833, 108.0000),
    'Dak Lak': (12.6667, 108.0333), 'Dak Nong': (12.0000, 107.6833),
    'Lam Dong': (11.9404, 108.4383),
    
    # Southeast (Dong Nam Bo)
    'Ho Chi Minh': (10.8231, 106.6297), 'Ba Ria Vung Tau': (10.5417, 107.2429),
    'Ba Ria - Vung Tau': (10.5417, 107.2429), 'Binh Duong': (11.1542, 106.6667),
    'Binh Phuoc': (11.7512, 106.9030), 'Dong Nai': (10.9574, 106.8427),
    'Tay Ninh': (11.3128, 106.0985),
    
    # Mekong River Delta (Cuu Long)
    'Long An': (10.5356, 106.4109), 'Dong Thap': (10.4579, 105.6732),
    'An Giang': (10.5255, 105.4255), 'Tien Giang': (10.3814, 106.2353),
    'Vinh Long': (10.2537, 105.9723), 'Ben Tre': (10.2430, 106.3756),
    'Kien Giang': (10.0121, 105.0809), 'Can Tho': (10.0452, 105.7469),
    'Hau Giang': (9.7829, 105.4722), 'Soc Trang': (9.6037, 105.9743),
    'Bac Lieu': (9.2940, 105.7278), 'Ca Mau': (9.1769, 105.1524),
    'Tra Vinh': (9.9369, 106.3411)
}

In [15]:
num_provinces = len(province_coords)
print(num_provinces)

65


In [19]:
unique_provinces = list(province_coords.keys())
print(unique_provinces)

['Ha Noi', 'Ha Nam', 'Hai Duong', 'Hai Phong', 'Hung Yen', 'Nam Dinh', 'Ninh Binh', 'Thai Binh', 'Vinh Phuc', 'Bac Ninh', 'Ha Giang', 'Cao Bang', 'Bac Kan', 'Tuyen Quang', 'Lao Cai', 'Dien Bien', 'Lai Chau', 'Son La', 'Yen Bai', 'Hoa Binh', 'Thai Nguyen', 'Lang Son', 'Bac Giang', 'Phu Tho', 'Thanh Hoa', 'Nghe An', 'Ha Tinh', 'Quang Binh', 'Quang Tri', 'Thua Thien Hue', 'Da Nang', 'Quang Nam', 'Quang Ngai', 'Binh Dinh', 'Phu Yen', 'Khanh Hoa', 'Ninh Thuận', 'Ninh Thuan', 'Binh Thuận', 'Binh Thuan', 'Kon Tum', 'Gia Lai', 'Dak Lak', 'Dak Nong', 'Lam Dong', 'Ho Chi Minh', 'Ba Ria Vung Tau', 'Ba Ria - Vung Tau', 'Binh Duong', 'Binh Phuoc', 'Dong Nai', 'Tay Ninh', 'Long An', 'Dong Thap', 'An Giang', 'Tien Giang', 'Vinh Long', 'Ben Tre', 'Kien Giang', 'Can Tho', 'Hau Giang', 'Soc Trang', 'Bac Lieu', 'Ca Mau', 'Tra Vinh']


In [20]:
# Convert both to sets of unique names
df_provinces = set(clean_soil_df['province_name'].unique())
dict_provinces = set(province_coords.keys())

# Provinces in your dictionary, but missing from your DataFrame
extra_in_dict = dict_provinces - df_provinces
print("In province_coords, but missing from DataFrame:", extra_in_dict)

# Provinces in your DataFrame, but missing from your dictionary
extra_in_df = df_provinces - dict_provinces
print("In DataFrame, but missing from province_coords:", extra_in_df)

In province_coords, but missing from DataFrame: {'Binh Thuận', 'Ha Noi', 'Ba Ria - Vung Tau', 'Ninh Thuận', 'Ba Ria Vung Tau'}
In DataFrame, but missing from province_coords: {'Ba Ria-Vung Tau', 'Quang Ninh', 'Ha Noi\t'}


In [21]:
# Strip whitespace/tabs from the DataFrame column
clean_soil_df['province_name'] = clean_soil_df['province_name'].str.strip()

# Standardize Ba Ria-Vung Tau formats in the DataFrame to match your preferred format
clean_soil_df['province_name'] = clean_soil_df['province_name'].replace({
    'Ba Ria-Vung Tau': 'Ba Ria - Vung Tau', 
    'Ba Ria Vung Tau': 'Ba Ria - Vung Tau'
})

## Verify and Merge

In [22]:
# Convert unique values to sets
df_provinces = set(clean_soil_df['province_name'].unique())
dict_provinces = set(province_coords.keys())

# Check for any remaining mismatches
missing_in_dict = df_provinces - dict_provinces
missing_in_df = dict_provinces - df_provinces

print("Unmatched in dictionary:", missing_in_dict)
print("Unmatched in DataFrame:", missing_in_df)

# Compare total counts
print(f"Unique provinces in DataFrame: {len(df_provinces)}")
print(f"Unique provinces in dictionary: {len(dict_provinces)}")

Unmatched in dictionary: {'Quang Ninh'}
Unmatched in DataFrame: {'Ninh Thuận', 'Ba Ria Vung Tau', 'Binh Thuận'}
Unique provinces in DataFrame: 63
Unique provinces in dictionary: 65


In [23]:
clean_soil_df['province_name'] = clean_soil_df['province_name'].replace({
    'Ninh Thuan': 'Ninh Thuận',
    'Binh Thuan': 'Binh Thuận',
    'Ba Ria-Vung Tau': 'Ba Ria Vung Tau'
})

province_coords['Quang Ninh'] = (21.0069, 107.2917)

In [24]:
print(set(clean_soil_df['province_name'].unique()) - set(province_coords.keys()))

set()


In [25]:
# Map the dictionary to get tuples of (latitude, longitude)
coords_series = clean_soil_df['province_name'].map(province_coords)

# Split the tuple into separate latitude and longitude columns
clean_soil_df['latitude'] = coords_series.str[0]
clean_soil_df['longitude'] = coords_series.str[1]

In [26]:
missing_coords = clean_soil_df[clean_soil_df['latitude'].isna()]['province_name'].unique()
print("Provinces missing coordinates:", missing_coords)

Provinces missing coordinates: <ArrowStringArray>
[]
Length: 0, dtype: str


In [28]:
clean_soil_df.head()

,province_name,soil,latitude,longitude
0,Ba Ria - Vung Tau,Luvisols (High-Nutrient Clay),10.5417,107.2429
1,Khanh Hoa,Other Mineral Soils,12.2388,109.1967
2,Phu Yen,Other Mineral Soils,13.0884,109.3097
3,Binh Dinh,Dystric Gleysols (Alluvial),13.7820,109.2197
4,Hai Phong,Orthic Acrisols (Acid Red-Yellow),20.8449,106.6881


## Add static features data (GEE)

In [81]:
static_features_df = pd.read_csv("../data/raw/soil/static_features_raw.csv", encoding='utf-8-sig')

In [82]:
static_features_df.head()

,annual_precip_mm,elevation_m,mean_temp_c,shapeGroup,shapeID,province_name,shapeType,slope_deg,soil_clay_pct,soil_ph,soil_soc
0,1434.337443,9.283938,27.376882,VNM,63759600B46141220358660,An Giang,ADM1,0.751189,36.297106,5.652158,37.336092
1,1969.896734,50.333049,26.460736,VNM,63759600B21811289888982,Bà R?a?V?ng Tàu,ADM1,1.412135,33.482474,5.938095,34.284765
2,1560.342226,130.287500,22.953978,VNM,63759600B37262939240335,B?c Giang,ADM1,5.491901,31.430396,5.908120,30.425907
3,1666.510323,488.980609,21.126883,VNM,63759600B29118141744967,B?c K?n,ADM1,11.895694,32.624861,5.487377,40.243328
4,2116.622676,1.982630,26.696532,VNM,63759600B49404698723526,B?c Liêu,ADM1,0.220902,40.618068,5.589189,42.332708


In [83]:
static_features_df.columns

Index(['annual_precip_mm', 'elevation_m', 'mean_temp_c', 'shapeGroup',
       'shapeID', 'province_name', 'shapeType', 'slope_deg', 'soil_clay_pct',
       'soil_ph', 'soil_soc'],
      dtype='str')

In [84]:
# Check total missing values per column
missing_counts = static_features_df.isnull().sum()
print(missing_counts[missing_counts > 0]) # Shows only columns with missing data

# Or get a complete overview
static_features_df.info()

soil_clay_pct    1
soil_ph          1
soil_soc         1
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   annual_precip_mm  64 non-null     float64
 1   elevation_m       64 non-null     float64
 2   mean_temp_c       64 non-null     float64
 3   shapeGroup        64 non-null     str    
 4   shapeID           64 non-null     str    
 5   province_name     64 non-null     str    
 6   shapeType         64 non-null     str    
 7   slope_deg         64 non-null     float64
 8   soil_clay_pct     63 non-null     float64
 9   soil_ph           63 non-null     float64
 10  soil_soc          63 non-null     float64
dtypes: float64(7), str(4)
memory usage: 8.1 KB


In [85]:
# Find rows where any of these three columns are NaN
missing_rows = static_features_df[
    static_features_df['soil_clay_pct'].isna() | 
    static_features_df['soil_ph'].isna() | 
    static_features_df['soil_soc'].isna()
]

print(missing_rows[['province_name', 'soil_clay_pct', 'soil_ph', 'soil_soc']])

   province_name  soil_clay_pct  soil_ph  soil_soc
14       Côn ??o            NaN      NaN       NaN


In [86]:
static_features_df = static_features_df.dropna(subset=['soil_clay_pct', 'soil_ph', 'soil_soc'])

In [87]:
# Apply unidecode directly to the original province names
static_features_df['province_name'] = static_features_df['province_name'].apply(unidecode)

# Standardize any hyphens or spaces if needed
static_features_df['province_name'] = static_features_df['province_name'].str.strip()

In [88]:
# 1. Ensure both dataframes are sorted identically (e.g., alphabetically by shapeID or original order)
static_features_df = static_features_df.sort_values('shapeID').reset_index(drop=True)
clean_soil_df = clean_soil_df.sort_values('province_name').reset_index(drop=True) # or keep original index if order matches

# 2. Directly inject the correct, clean province names and columns from your clean dictionary/dataframe
static_features_df['province_name'] = clean_soil_df['province_name']
static_features_df['soil'] = clean_soil_df['soil']
static_features_df['latitude'] = clean_soil_df['latitude']
static_features_df['longitude'] = clean_soil_df['longitude']

# 3. Verify zero missing values remain
print("Missing values in master dataset:")
print(static_features_df[['province_name', 'soil', 'latitude', 'longitude']].isna().sum())

Missing values in master dataset:
province_name    0
soil             0
latitude         0
longitude        0
dtype: int64


In [90]:
# Drop shapeGroup, shapeID, and shapeType columns
static_features_df = static_features_df.drop(columns=['shapeGroup', 'shapeID', 'shapeType'])

# Verify the updated column list
print(static_features_df.columns)

Index(['annual_precip_mm', 'elevation_m', 'mean_temp_c', 'province_name',
       'slope_deg', 'soil_clay_pct', 'soil_ph', 'soil_soc', 'soil', 'latitude',
       'longitude'],
      dtype='str')


In [91]:
static_features_df.head()

,annual_precip_mm,elevation_m,mean_temp_c,province_name,slope_deg,soil_clay_pct,soil_ph,soil_soc,soil,latitude,longitude
0,1527.831878,627.710489,20.539134,An Giang,12.683626,32.512793,5.635906,38.022836,Other Mineral Soils,10.5255,105.4255
1,1386.415155,2.873666,27.371122,Ba Ria - Vung Tau,0.220305,39.614501,6.319651,40.048719,Luvisols (High-Nutrient Clay),10.5417,107.2429
2,2193.704263,102.793618,26.007287,Bac Giang,1.540575,31.542732,5.555299,31.956809,Acrisols (Red-Yellow / Sloped),21.2731,106.1947
3,2358.683061,176.889841,25.575723,Bac Kan,2.555262,32.457602,5.303649,30.726536,Other Mineral Soils,22.1500,105.8333
4,1870.583012,532.765435,23.651343,Bac Lieu,5.460301,32.272266,5.764224,35.490836,Gleysols (Lowland Alluvial / Wet),9.2940,105.7278


In [92]:
static_features_df.to_csv('soil_climate_data.csv', index=False)

print("File saved as soil_climate_data.csv!")

File saved as soil_climate_data.csv!
